In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [2]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

CALIBRATE_DATASET_PATH = os.path.join(DATASETS_PATH, "calibrate-dnam")
X_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "features.dnam.csv")
y_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "labels.csv")

TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "test-dnam")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.dnam.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")

output_dir = "/home/mariya/Data/pallas/models/dnam"
random_seed = 42
n_jobs = 10
rf_model_path = os.path.join(output_dir, "rf_model.pkl")
selected_features_path = os.path.join(output_dir, "selected_features.txt")
pipeline_path = os.path.join(output_dir, "pipeline.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_calibrate_path: {X_calibrate_path}")
print(f"y_calibrate_path: {y_calibrate_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"rf_model_path: {rf_model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

# Hardcoded variables
FEATURE_SELECTION_PERCENTILE=10  # KEEP TOP n% OF CPG SITES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-dnam/features.dnam.csv
y_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-dnam/labels.csv
X_calibrate_path: /home/mariya/Data/pallas/mlomix_datasets/calibrate-dnam/features.dnam.csv
y_calibrate_path: /home/mariya/Data/pallas/mlomix_datasets/calibrate-dnam/labels.csv
X_test_path: /home/mariya/Data/pallas/mlomix_datasets/test-dnam/features.dnam.csv
y_test_path: /home/mariya/Data/pallas/mlomix_datasets/test-dnam/labels.csv
output_dir: /home/mariya/Data/pallas/models/dnam
random_seed: 42
n_jobs: 10
rf_model_path: /home/mariya/Data/pallas/models/dnam/rf_model.pkl
selected_features_path: /home/mariya/Data/pallas/models/dnam/selected_features.txt
pipeline_path: /home/mariya/Data/pallas/models/dnam/pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/models/dnam/hyperparams.pkl


In [ ]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


In [ ]:
y_train = pd.read_csv(y_train_path, index_col=0)
y_train

,class
id,
ALL_1,"B-ALL, high hyperdiploid"
ALL_1002,"B-ALL, high hyperdiploid"
ALL_1004,"B-ALL, PAX5alt"
ALL_1007,"B-ALL, high hyperdiploid"
ALL_1008,"B-ALL, high hyperdiploid"
...,...
TCGA-AB-2997,"AML, APL-PML::RARA"
TCGA-AB-2999,"AML, APL-PML::RARA"
TCGA-AB-3001,"AML, APL-PML::RARA"


In [ ]:
# Show class distributions in y_train and y_test
print("y_train class distribution:")
print(y_train.value_counts())


y_train class distribution:
class                   
AML, KMT2A-r                246
B-ALL, high hyperdiploid    229
B-ALL, ETV6::RUNX1          199
AML, RUNX1::RUNX1T1         173
AML, NPM1alt                131
AML, CBFB::MYH11            126
Non-leukemic                 95
T-ALL                        73
AML, NUP98-r                 68
AML, CEBPAalt                55
B-ALL, KMT2A-r               43
B-ALL, PAX5alt               35
B-ALL, TCF3::PBX1            26
AML, CBFA2T3::GLIS2          22
AML, DEK::NUP214             19
B-ALL, BCR::ABL1             17
AML, APL-PML::RARA           16
AML, ETV6-r                  16
B-ALL, iAMP21                15
B-ALL, DUX4-r                14
B-ALL, ZNF384-r              12
B-ALL, ETV6::RUNX1-like      10
B-ALL, BCR::ABL1-like         8
AML, KAT6A-r                  8
AML, RBM15::MKL1              8
B-ALL, hypodiploid            7
B-ALL, PAX5 P80R              6
AML, FUS::ERG                 6
B-ALL, MEF2D-r                6
AML, MECOM-r       

In [ ]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [ ]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array

In [ ]:
# Reduce memory usage by converting to float32
X_train = X_train.astype(np.float32)

In [19]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS
N_FOLDS_INITIAL=3 # down from min_class_count to speed up initial search

# ----------------------
# Pipeline: RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs)) # inner: RF uses all cores
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50],  # We will increase this in the final model
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2', 0.25],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': ['balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=N_FOLDS_INITIAL,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=1, # outer loop: sequential
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparams_path)

# get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = X_train.columns[selected_mask].tolist()

# Write selected features to a text file
with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters: {'rf__n_estimators': 50, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 1, 'rf__max_features': 0.25, 'rf__max_depth': 20, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.8811226235411319


['/home/mariya/Data/pallas/models/dnam/pipeline.pkl']

In [20]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparams_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=1,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.25, 'max_depth': 20, 'class_weight': 'balanced'}


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed: 395.2min
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed: 402.9min finished


Cross-validation F1-weighted: 0.891 ± 0.014


['/home/mariya/Data/pallas/models/dnam/final_model.pkl']

In [21]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)

# Read in X_calibrate and y_calibrate and append them to test
X_calibrate = pd.read_csv(X_calibrate_path, index_col=0)
y_calibrate = pd.read_csv(y_calibrate_path, index_col=0)
X_test = pd.concat([X_test, X_calibrate], axis=0)
y_test = pd.concat([y_test, y_calibrate], axis=0)

y_test

,class
id,
ALL_100,"B-ALL, high hyperdiploid"
ALL_1029,"B-ALL, high hyperdiploid"
ALL_1064,"B-ALL, high hyperdiploid"
ALL_1068,"B-ALL, KMT2A-r"
ALL_1076,"B-ALL, KMT2A-r"
...,...
TCGA-AB-2984,"AML, NPM1alt"
TCGA-AB-2986,"AML, NPM1alt"
TCGA-AB-2998,"AML, APL-PML::RARA"


In [22]:
print("\ny_test class distribution:")
print(y_test.value_counts())


y_test class distribution:
class                   
AML, KMT2A-r                105
B-ALL, high hyperdiploid     98
B-ALL, ETV6::RUNX1           85
AML, RUNX1::RUNX1T1          75
AML, NPM1alt                 56
AML, CBFB::MYH11             54
Non-leukemic                 41
T-ALL                        32
AML, NUP98-r                 30
AML, CEBPAalt                23
B-ALL, KMT2A-r               18
B-ALL, PAX5alt               15
B-ALL, TCF3::PBX1            11
AML, CBFA2T3::GLIS2          10
AML, DEK::NUP214              8
B-ALL, BCR::ABL1              8
B-ALL, iAMP21                 7
AML, ETV6-r                   7
AML, APL-PML::RARA            7
B-ALL, DUX4-r                 6
B-ALL, ZNF384-r               5
B-ALL, ETV6::RUNX1-like       4
B-ALL, BCR::ABL1-like         3
B-ALL, MEF2D-r                3
AML, RBM15::MKL1              3
AML, KAT6A-r                  3
B-ALL, hypodiploid            3
AML, MECOM-r                  2
AML, FUS::ERG                 2
B-ALL, PAX5 P80R   

In [23]:

y_test = y_test.iloc[:, 0].values


In [24]:
# ----------------------
# Evaluate final model on test set
# ----------------------

X_test_selected = X_test[selected_features]

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 18191
Shape of X_test_selected: (726, 18191)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     0.857     0.923         7
     AML, CBFA2T3::GLIS2      1.000     0.800     0.889        10
        AML, CBFB::MYH11      0.964     0.981     0.972        54
           AML, CEBPAalt      1.000     0.957     0.978        23
        AML, DEK::NUP214      1.000     0.875     0.933         8
             AML, ETV6-r      0.857     0.857     0.857         7
           AML, FUS::ERG      1.000     1.000     1.000         2
            AML, KAT6A-r      1.000     0.333     0.500         3
            AML, KMT2A-r      0.806     0.952     0.873       105
            AML, MECOM-r      0.000     0.000     0.000         2
            AML, NPM1alt      0.940     0.839     0.887        56
            AML, NUP98-r      0.947     0.600     0.735        30
        AM

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [25]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
             Gene  Importance
4857   cg06623668    0.012113
13     cg00015639    0.007003
5069   cg06891858    0.006603
476    cg00587115    0.006557
12850  cg18638389    0.005896
12074  cg17411792    0.004822
325    cg00424467    0.004547
14658  cg21912448    0.004507
5351   cg07232063    0.004505
6006   cg08166720    0.004129
10589  cg14992144    0.004095
1462   cg01912455    0.004031
17599  cg26786253    0.003830
16675  cg25287071    0.003634
8660   cg12183594    0.003531
301    cg00393348    0.003240
2983   cg03930153    0.003216
4508   cg06078469    0.003195
6226   cg08457158    0.003080
8733   cg12338894    0.003024


In [26]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 4997 out of 18191

Bottom 20 genes by importance (final model):
             Gene  Importance
4584   cg06222851         0.0
4589   cg06237774         0.0
4591   cg06240450         0.0
4597   cg06248560         0.0
14105  cg20864608         0.0
14113  cg20879085         0.0
4566   cg06180869         0.0
4571   cg06188083         0.0
7972   cg11132534         0.0
4577   cg06202686         0.0
4580   cg06205746         0.0
14120  cg20891842         0.0
14091  cg20820063         0.0
14097  cg20842915         0.0
7959   cg11113760         0.0
7960   cg11116200         0.0
7964   cg11124364         0.0
7969   cg11130317         0.0
7971   cg11131532         0.0
4573   cg06191495         0.0
